In [ ]:
import os
import json
import io
import zipfile
import urllib.request
import boto3

def api(a, b):
    try:
        # Retrieve credentials und configuration from environment variables
        bucket_name = os.environ.get('S3_BUCKET_NAME')
        token = os.environ.get('KAGGLE_API_TOKEN')
        
        if not bucket_name or not token:
            raise Exception("Fehler: Die Sicherheitsvariablen fehlen!")

        # Exact Kaggle URL
        url = "https://www.kaggle.com/api/v1/datasets/download/olistbr/brazilian-ecommerce"

        # 1. API -request with urllib since request library was not available
        req = urllib.request.Request(url)
        req.add_header("Authorization", f"Bearer {token}")
        
        print("⏳ Starte Request ")
        # Open connection and read the zipped dataset into memory with a contextmanager
        with urllib.request.urlopen(req) as response:
            zip_content = response.read()

        # Initialize the AWS S3 client
        s3_client = boto3.client('s3')

        # Open the Zip file directly in ram using io.bytesIO and a contextmanager
        with zipfile.ZipFile(io.BytesIO(zip_content)) as zip_file:
            # zip_file.namelist() gibt uns eine Liste ALLER Dateinamen in der ZIP
            all_files = zip_file.namelist()
            print(f"📦 Gefundene Dateien in der ZIP: {all_files}")
            
            # 4. Checking the files with a loop
            for current_file in all_files:
                # Exclude empty files
                if current_file.endswith('/'):
                    continue
                
                print(f"🔄 Verarbeite Datei: {current_file}")
                
                # Reading packed files
                file_data = zip_file.read(current_file)
                #Define target path und prefix within the S3 Bucket
                s3_target_key = f"_raw/{current_file}"
                
                # Upload the file content directly to S3
                print(f"🚀 Upload zu S3: {s3_target_key}")
                s3_client.put_object(
                    Bucket=bucket_name,
                    Key=s3_target_key,
                    Body=file_data
                )
        
        print("✅ ALL files successfully transferred to S3")
        return {
            'statusCode': 200,
            'body': json.dumps('Success! Data pipeline completed securely.')
        }
        
    except Exception as e:
        print(f"❌ Pipeline leaked: {str(e)}")
        return {
            'statusCode': 500,
            'body': json.dumps(f'Fehler: {str(e)}')
        }
